# Protein BPE Tokenizer — PoC
**Goal**: Download 10,000 SwissProt sequences, train a BPE tokenizer (vocab_size=4096), and save it for later merging with the OPT tokenizer.

Pipeline:
1. Download sequences from UniProt (SwissProt)
2. Extract sequences → `sequences.txt`
3. Train BPE tokenizer
4. Inspect & save

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers requests biopython

## 2. Load protein sequences

In [ ]:
import pandas as pd

swissprot_file = '/run/media/khairi/seagate/data/swissprot/data/processed/swissprot_sequences_20_512.tsv'

df = pd.read_csv(swissprot_file, sep='\t')

sequences = df['Sequence'].tolist()

print(f'Loaded {len(sequences)} sequences')
with open('/tmp/sequences.txt', 'w') as f:
    for seq in sequences:
        f.write(f'{seq}\n')

In [ ]:
!head -5 /tmp/sequences.txt

## 4. Train BPE tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Split
from tokenizers import pre_tokenizers

# Build tokenizer
tokenizer = Tokenizer(BPE(unk_token=None))

trainer = BpeTrainer(
    vocab_size=4096,
    min_frequency=2,
    show_progress=True,
    special_tokens=[]   # no special tokens — handled at OPT tokenizer level
)

print("Training BPE tokenizer...")
tokenizer.train(files=["/tmp/sequences.txt"], trainer=trainer)
print(f"Vocab size: {tokenizer.get_vocab_size()}")

## 5. Sanity check

In [ ]:
test_seq = sequences[0][:20]
encoding = tokenizer.encode(test_seq)

print(f"Input  : {test_seq}")
print(f"Tokens : {encoding.tokens}")
print(f"IDs    : {encoding.ids}")
print(f"N tokens: {len(encoding.tokens)} for {len(test_seq)} chars")

## 6. Inspect vocabulary

In [ ]:
vocab = tokenizer.get_vocab()

# Sort by ID
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])

print("=== First 20 tokens (single amino acids + early merges) ===")
for token, idx in sorted_vocab[:20]:
    print(f"  {idx:5d} | {token}")

print("\n=== Last 10 tokens (longest merges) ===")
for token, idx in sorted_vocab[-10:]:
    print(f"  {idx:5d} | {token}")

## 7. Check collisions with Flan-T5 vocabulary (preview)

In [ ]:
from transformers import AutoTokenizer

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_vocab = set(flan_tokenizer.get_vocab().keys())
protein_vocab = set(vocab.keys())

collisions = flan_vocab & protein_vocab
print(f"FLAN-T5 vocab size    : {len(flan_vocab)}")
print(f"Protein vocab size: {len(protein_vocab)}")
print(f"Collisions        : {len(collisions)}")
print(f"\nSample collisions : {list(collisions)[:30]}")

## 8. Save protein tokenizer

In [ ]:
import os
os.makedirs("/tmp/protein_tokenizer", exist_ok=True)
tokenizer.save("/tmp/protein_tokenizer/tokenizer.json")
print("Saved to /tmp/protein_tokenizer/tokenizer.json")

In [ ]:
import json

with open("/tmp/protein_tokenizer/tokenizer.json") as f:
    protein_tok_data = json.load(f)

protein_vocab  = protein_tok_data["model"]["vocab"]   # dict: token -> id
protein_merges = protein_tok_data["model"]["merges"]  # list of "A B" strings

print(f"Protein vocab size : {len(protein_vocab)}")
print(f"Protein merges     : {len(protein_merges)}")
print(f"Sample tokens      : {list(protein_vocab.keys())[:20]}")
print(f"Sample merges      : {protein_merges[:10]}")

In [ ]:
flan_vocab = flan_tokenizer.get_vocab()  # dict: token -> id

new_tokens = [
    tok for tok in protein_vocab
    if tok not in flan_vocab
]

print(f"Flan-T5 vocab size : {len(flan_vocab)}")
print(f"Net-new tokens     : {len(new_tokens)}")
print(f"Sample new tokens  : {new_tokens[:20]}")

from transformers import AddedToken

added_token_objects = [
    AddedToken(tok, single_word=True, normalized=False, special=False)
    for tok in new_tokens
]

n_added = flan_tokenizer.add_tokens(added_token_objects)
print(f"Added {n_added} tokens")
print(f"New Flan-T5 vocab size: {len(flan_tokenizer)}")

In [ ]:
import json
from tokenizers.models import BPE

backend = flan_tokenizer.backend_tokenizer
backend_data = json.loads(backend.to_str())

print("Backend model type:", backend_data["model"]["type"])

In [ ]:
test_seq = "MKTAYIAKQR"

tokens = flan_tokenizer.tokenize(test_seq)
print(f"Tokenization of '{test_seq}':")
print(f"  {tokens}")
print(f"  ({len(tokens)} tokens for {len(test_seq)} chars)")

In [ ]:
import os
os.makedirs("/tmp/flan_t5_protein_extended", exist_ok=True)
flan_tokenizer.save_pretrained("/tmp/flan_t5_protein_extended")

print("Files saved:")
print(os.listdir("/tmp/flan_t5_protein_extended"))

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("/tmp/flan_t5_protein_extended")
print(f"Reloaded vocab size: {len(tok)}")

for t in new_tokens[:5]:
    tid = tok.convert_tokens_to_ids(t)
    assert tid != tok.unk_token_id, f"{t} resolved to <unk>!"
    print(f"  '{t}' → id {tid}  ✓")

In [ ]:
test_seqs = [
    "MKTAYIAKQR",
    "ACDEFGHIKLMNPQRSTVWY",
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGT",
]

print("=" * 60)
for seq in test_seqs:
    tokens = flan_tokenizer.tokenize(seq)
    ids    = flan_tokenizer.convert_tokens_to_ids(tokens)

    # how many tokens came from the new protein vocab
    n_protein = sum(1 for t in tokens if t in set(new_tokens))
    n_unk     = sum(1 for i in ids if i == flan_tokenizer.unk_token_id)

    print(f"Sequence : {seq[:44]}")
    print(f"Tokens   : {tokens}")
    print(f"IDs      : {ids}")
    print(f"Length   : {len(seq)} chars → {len(tokens)} tokens (compression {len(seq)/len(tokens):.2f}x)")
    print(f"Protein tokens used : {n_protein}/{len(tokens)}")
    print(f"UNK count           : {n_unk}  ← should be 0")
    print("-" * 60)

In [ ]:
from tqdm.notebook import tqdm
new_tokens_used = 0.0
for seq in tqdm(sequences):
    ids = tok.convert_tokens_to_ids(tok.tokenize(seq))
    for idx in ids:
        if idx > 32100:
            new_tokens_used += 1
print(f"New tokens usage: {new_tokens_used}")